Этот ноутбук демонстрирует технику Neural Style Transfer, предложенную в работе Gatys et al., 2015. 
[https://arxiv.org/abs/1508.06576?spm=a2ty_o01.29997173.0.0.2da951719zUPyO&file=1508.06576]

Суть метода — создать новое изображение, которое:
сохраняет содержание (content) одного изображения (например, фотографии),
но принимает стиль (style) другого изображения (например, картины Ван Гога).

 

Мы используем torch для вычислений, torchvision.transforms для предобработки изображений, PIL для загрузки, и matplotlib — для визуализации

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

import torch
from torchvision import models
import torchvision.transforms.v2 as tfs_v2
import torch.nn as nn
import torch.optim as optim

In [ ]:
img = Image.open('../pics/image.jpg').convert('RGB')
img_style = Image.open('../pics/image_style.jpg').convert('RGB')

In [ ]:
plt.imshow(img)

In [ ]:
plt.imshow(img_style)

Далее, планируется использовать сеть VGG19, поэтому наши изображения нужно преобразовать в тензор фреймворка PyTorch. Для этого определим следующее преобразование:

In [ ]:
# transforms = models.VGG19_Weights.DEFAULT.transforms()
transforms = tfs_v2.Compose([tfs_v2.ToImage(),
                             tfs_v2.ToDtype(torch.float32, scale=True),
                             ])
# И применим его к загруженным изображениям:
img = transforms(img).unsqueeze(0)
img_style = transforms(img_style).unsqueeze(0)


Напомню, что метод unsqueeze(0) необходим для добавления первой оси – размер батча входного тензора.

После этого создадим тензор для формируемого изображения (то, которое будет непосредственно подвергаться стилизации):

In [ ]:

img_create = img.clone()
img_create.requires_grad_(True)


Обратите внимание, что у тензора img_create необходимо включить градиенты, для изменения пикселей градиентным алгоритмом.

Далее нам потребуется обученная модель сети VGG19, но без полносвязной НС. В данной задаче она нам не нужна. Мы уже знаем, что формально такую модель можно определить командами:

In [ ]:
model = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
mf = model.features

In [ ]:


class ModelStyle(nn.Module):
    def __init__(self):
        super().__init__()
        _model = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        self.mf = _model.features
        self.mf.requires_grad_(False)
        self.requires_grad_(False)
        self.mf.eval()
        self.idx_out = (0, 5, 10, 19, 28, 34)
        self.num_style_layers = len(self.idx_out) - 1 # последний слой для контента

    def forward(self, x):
        outputs = []
        for indx, layer in enumerate(self.mf):
            x = layer(x)
            if indx in self.idx_out:
                outputs.append(x.squeeze(0))

        return outputs



 В инициализаторе создается внутренняя модель VGG19 с набором обученных весов, и оставляются только сверточные слои. 
 
 При этом модель self.mf впоследствии не должна обучаться и градиенты для ее весовых коэффициентов вычислять не нужно. 
 
 Поэтому они отключаются командой self.mf.requires_grad_(False). 
 
 То же самое делается и для всей текущей модели. Далее, модель переводится в режим эксплуатации и формируется набор индексов слоев, с которых будут сниматься данные. Почему именно такие индексы? Если распечатать модель self.mf, то мы увидим следующую информацию:

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)

  (br): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
…

На ее основе и были сформирован список из индексов нужных нам слоев. Причем, последний индекс 34 относится к последнему сверточному слою, по которому будут определяться потери по контенту, а все остальные индексы – к слоям для вычисления потерь по стилизации.

Локальная переменная self.num_style_layers – это вспомогательная переменная, хранящая число сверточных слоев используемых для оценки степени соответствия стилизации.

В методе forward входной тензор x прогоняется по всем слоям модели self.mf и если индекс текущего слоя содержится в списке self.idx_out, то его данные сохраняются в списке outputs. В конце этот список возвращается.

Все, теперь у нас есть нужная нам модель, дающая всю необходимую информацию. Создадим ее командой:

In [ ]:

model = ModelStyle()




и пропустим через нее оба загруженных изображения:

In [ ]:
outputs_img = model(img)
outputs_img_style = model(img_style)

In [ ]:
outputs_img_create = model(img_create)

In [ ]:
def get_content_loss(base_content, target):
    return torch.mean( torch.square(base_content - target) )

In [ ]:

# Матрица Грама — это корреляция между каналами признаков. Она кодирует стиль, но не пространственную структуру.
def gram_matrix(x):
  channels = x.size(dim=0)
  g = x.view(channels, -1)
  gram = torch.mm(g, g.mT) / g.size(dim=1)
  return gram

In [ ]:
def get_style_loss(base_style, gram_target):
    style_weights = [1.0, 0.8, 0.5, 0.3, 0.1]

    _loss = 0
    i = 0
    for base, target in zip(base_style, gram_target):
        gram_style = gram_matrix(base)
        _loss += style_weights[i] * torch.mean(torch.square(gram_style - target))
        i += 1

    return _loss

In [ ]:
gram_matrix_style = [gram_matrix(x) for x in outputs_img_style[:model.num_style_layers]]

Будем сохранять лучшие результаты стилизации с помощью переменных best_loss и best_img, установим 100 эпох для обучения и воспользуемся оптимизатором Adam:

In [ ]:

content_weight = 1
style_weight = 1000
best_loss = -1
epochs = 100

optimizer = optim.Adam(params=[img_create], lr=0.01)
best_img = img_create.clone()

Обратите внимание, что оптимизируемые параметры – это пиксели формируемого изображения.

Осталось описать главный цикл обучения. Реализуем его следующим образом:

In [ ]:


for _e in range(epochs):
    outputs_img_create = model(img_create)

    loss_content = get_content_loss(outputs_img_create[-1], outputs_img[-1])
    loss_style = get_style_loss(outputs_img_create, gram_matrix_style)
    loss = content_weight * loss_content + style_weight * loss_style

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    img_create.data.clamp_(0, 1)

    if loss < best_loss or best_loss < 0:
      best_loss = loss
      best_img = img_create.clone()

    print(f'Iteration: {_e}, loss: {loss.item(): .4f}')



Формируемое изображение пропускается через модель. 

Затем, вычисляются потери по контенту и стилю. Формируется общее значение потерь.

И для их минимизации делается один шаг градиентного спуска, меняя пиксели изображения img_create. 

После этого значения тензора img_create ограничиваются исходным диапазоном [0; 1]. 

Далее идет проверка на определение лучшего варианта стилизации с точки зрения функции потерь.

Лучшее изображение сохраняется в переменной best_img и в конце выводится информация о текущей эпохе и значение функции потерь.

Чтобы визуально оценить полученный результат, выведем лучшее полученное изображение на экран. Для этого подготовим изображение best_img к отображению:

In [ ]:
x = best_img.detach().squeeze()
low, hi = torch.amin(x), torch.amax(x)
x = (x - low) / (hi - low) * 255.0
x = x.permute(1, 2, 0)
x = x.numpy()
x = np.clip(x, 0, 255).astype('uint8')



In [ ]:
Сохраним его в выходной файл:

In [ ]:
image = Image.fromarray(x, 'RGB')
image.save("result.jpg")

# И ВЫВЕДЕМ НА ЭКРАН
print(best_loss)
plt.imshow(x)
plt.show()